# Step 3 — Train & Save the Final Deployable Model

**Why this notebook is different from notebook 2:** notebook 2 trained a *separate* LSTM per student, which is great for comparing model types but useless for deployment — a real app user isn't "Student 7."

Here we train **one single "global" LSTM**, pooled across all 10 students' training data, so it learns *general* patterns (how a stress/mental-health score tends to evolve given the last 30 days) rather than memorizing one person. This is the model your FastAPI backend will actually load and serve.

**Output of this notebook:**
- `model_lstm_global.keras` — the trained model
- `scaler.save` — the scaler needed to preprocess new input the same way
- A tested `predict_90_days()` function that takes "a student's last 30 days of scores" → returns a 90-day forecast + category labels


## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import joblib

from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import LSTM, Dense
from tensorflow.keras.callbacks import EarlyStopping

import warnings
warnings.filterwarnings("ignore")

np.random.seed(42)
import tensorflow as tf
tf.random.set_seed(42)


## 2. Load data & config

Same TARGET, HORIZON, LOOK_BACK as notebook 2 — keep these consistent.

In [ ]:
DATA_PATH = "../data/processed/student_data_clean.csv"

df = pd.read_csv(DATA_PATH, parse_dates=['Timestamp'])
df = df.sort_values(['StudentID', 'Timestamp']).reset_index(drop=True)

TARGET = "Mental_Health_Status_Score"
HORIZON = 90
LOOK_BACK = 30
STUDENT_IDS = sorted(df['StudentID'].unique())

mh_order = ['Normal', 'Mild Stress', 'Moderate Stress', 'Severe Stress', 'Anxiety', 'Depression']

def get_series(sid, target=TARGET):
    return (df[df['StudentID'] == sid]
            .sort_values('Timestamp')
            .set_index('Timestamp')[target]
            .astype(float))


## 3. Build the pooled training set

We use only each student's **training portion** (everything except their last 90 days) — exactly the same split as notebook 2 — so this final model is evaluated fairly and doesn't "cheat" by seeing test-period data.

In [ ]:
def make_windows(values, look_back=LOOK_BACK):
    X, y = [], []
    for i in range(len(values) - look_back):
        X.append(values[i:i + look_back])
        y.append(values[i + look_back])
    return np.array(X), np.array(y)

# Fit ONE scaler on all pooled training data (not per-student) so the deployed model
# sees the same scaling logic no matter which real user's data comes in later.
all_train_values = []
for sid in STUDENT_IDS:
    s = get_series(sid)
    train = s.iloc[:-HORIZON]
    all_train_values.append(train.values)
all_train_values = np.concatenate(all_train_values).reshape(-1, 1)

scaler = MinMaxScaler()
scaler.fit(all_train_values)

X_pool, y_pool = [], []
for sid in STUDENT_IDS:
    s = get_series(sid)
    train = s.iloc[:-HORIZON]
    scaled = scaler.transform(train.values.reshape(-1, 1)).flatten()
    X, y = make_windows(scaled)
    X_pool.append(X)
    y_pool.append(y)

X_pool = np.concatenate(X_pool)
y_pool = np.concatenate(y_pool)
X_pool = X_pool.reshape((X_pool.shape[0], X_pool.shape[1], 1))

print("Pooled training windows:", X_pool.shape, y_pool.shape)


## 4. Build and train the global LSTM

In [ ]:
def build_lstm(look_back=LOOK_BACK):
    model = Sequential([
        LSTM(32, input_shape=(look_back, 1)),
        Dense(16, activation='relu'),
        Dense(1)
    ])
    model.compile(optimizer='adam', loss='mse')
    return model

model = build_lstm()
es = EarlyStopping(monitor='loss', patience=5, restore_best_weights=True)

history = model.fit(
    X_pool, y_pool,
    epochs=50,
    batch_size=64,
    verbose=1,
    callbacks=[es]
)


## 5. Sanity-check on held-out data

Before trusting this model, check it still performs reasonably on each student's held-out last-90-days test period (same test sets as notebook 2) — this should be roughly in line with (not necessarily identical to, since this is now one shared model instead of 10 individual ones) the per-student LSTM RMSE you already saw.

In [ ]:
from sklearn.metrics import mean_squared_error

def recursive_forecast(model, last_known_values, horizon=HORIZON, look_back=LOOK_BACK):
    scaled = scaler.transform(np.array(last_known_values).reshape(-1, 1)).flatten()
    window = list(scaled[-look_back:])
    preds_scaled = []
    for _ in range(horizon):
        x_input = np.array(window[-look_back:]).reshape((1, look_back, 1))
        next_val = model.predict(x_input, verbose=0)[0, 0]
        preds_scaled.append(next_val)
        window.append(next_val)
    preds_scaled = np.array(preds_scaled).reshape(-1, 1)
    return scaler.inverse_transform(preds_scaled).flatten()

check_rmse = []
for sid in STUDENT_IDS:
    s = get_series(sid)
    train, test = s.iloc[:-HORIZON], s.iloc[-HORIZON:]
    pred = recursive_forecast(model, train.values)
    rmse = np.sqrt(mean_squared_error(test.values, pred))
    check_rmse.append(rmse)
    print(f"Student {sid}: global-model RMSE = {rmse:.3f}")

print(f"\nAverage global-model RMSE: {np.mean(check_rmse):.3f}")
print("(Compare this to the per-student LSTM average RMSE from notebook 2's model_comparison_results.csv)")


## 6. Convert numeric forecast → app category

Your app needs to display words like "Depression", not a raw number like `4.83`. We round the predicted score to the nearest valid class and map it back through the same ordered list used in notebook 1.

In [ ]:
def score_to_category(score, order=mh_order):
    idx = int(round(score))
    idx = max(0, min(idx, len(order) - 1))  # clip to valid range
    return order[idx]

def predict_90_days(recent_30_days_scores):
    """
    recent_30_days_scores: list/array of the last 30 days of Mental_Health_Status_Score
                            (as floats 0-5, from that user's own logged history)
    Returns: dict with the raw 90-day forecast and the category for day 90 (3 months out)
    """
    assert len(recent_30_days_scores) >= LOOK_BACK, f"Need at least {LOOK_BACK} days of history"
    forecast = recursive_forecast(model, recent_30_days_scores)
    return {
        "forecast_scores": forecast.tolist(),
        "day_30_category": score_to_category(forecast[29]),
        "day_60_category": score_to_category(forecast[59]),
        "day_90_category": score_to_category(forecast[89]),
    }


## 7. Test it end-to-end

Simulate a real API call: take one student's most recent 30 real days as "a user's history," and see what the function returns.

In [ ]:
sample_student = STUDENT_IDS[0]
s = get_series(sample_student)
recent_30 = s.iloc[-(HORIZON + LOOK_BACK):-HORIZON].values  # 30 days right before their test period

result = predict_90_days(recent_30)
print("3-month-ahead forecast for a sample user:")
print(f"  In 30 days -> {result['day_30_category']}")
print(f"  In 60 days -> {result['day_60_category']}")
print(f"  In 90 days -> {result['day_90_category']}")


## 8. Save the model and scaler

These two files are everything your FastAPI backend needs to make predictions — no retraining required at request time.

In [ ]:
model.save("model_lstm_global.keras")
joblib.dump(scaler, "scaler.save")
print("Saved model_lstm_global.keras and scaler.save")


## 9. Reload check

Confirms the saved files actually work when loaded fresh — exactly how your API will use them.

In [ ]:
loaded_model = load_model("model_lstm_global.keras")
loaded_scaler = joblib.load("scaler.save")

scaled_check = loaded_scaler.transform(recent_30.reshape(-1, 1)).flatten()
print("Reload successful. Scaled sample input (first 5 values):", scaled_check[:5])


## What's next

Move `model_lstm_global.keras` and `scaler.save` into a new `models/` folder in your project — the FastAPI backend (next step) will load them from there.

Next: I'll build the FastAPI `/predict` endpoint that wraps `predict_90_days()` so your Flutter app can call it over HTTP.
